# CasaScript 2.0 - Estufa Inteligente
Projeto da Prática 2 (Aula CP 06) - Compiladores
Grupo:
- Guilherme Pedigone - RA: 2712633
- Laryssa Dantas - RA: 2342968
- Gabriel Diogo - RA: 2683210

Cenário escolhido: **estufa / horta inteligente**.

A gente pegou o compilador da CasaScript feito em aula e adaptou para a estufa. As mudanças de cada requisito estão marcadas no código com comentários `# RF1`, `# RF2` etc. Mais detalhes no README.

## Etapa 1 - Preparação do ambiente

In [1]:
# ⚙️ CÉLULA 0 — Preparação do ambiente
# No Google Colab: basta executar esta célula.
# No VS Code: rode no terminal ->  pip install lark ipywidgets ipykernel
%pip install -q lark ipywidgets

import html, time, difflib, operator
from dataclasses import dataclass, field

import lark
from lark import Lark, Transformer, v_args
from lark.exceptions import UnexpectedCharacters, UnexpectedToken, UnexpectedEOF

import ipywidgets as widgets
from IPython.display import display, HTML

print("✅ Ambiente pronto! Lark versão", lark.__version__)

✅ Ambiente pronto! Lark versão 1.3.1


Funções só para mostrar tabelas e mensagens (iguais às da aula). Acrescentei a `hhmm()` para mostrar os horários.

In [2]:
# 🎨 CÉLULA 1 — Ferramentas de visualização (não fazem parte do compilador)

def tabela(cabecalho, linhas, titulo=None):
    """Mostra uma lista de linhas como tabela HTML."""
    th = "".join(f"<th style='padding:4px 12px;border-bottom:2px solid #888;text-align:left'>{c}</th>"
                 for c in cabecalho)
    corpo = ""
    for linha in linhas:
        tds = "".join(f"<td style='padding:3px 12px;border-bottom:1px solid #ccc;font-family:monospace;text-align:left'>"
                      f"{html.escape(str(c))}</td>" for c in linha)
        corpo += f"<tr>{tds}</tr>"
    topo = f"<b>{titulo}</b>" if titulo else ""
    display(HTML(f"{topo}<table style='border-collapse:collapse;margin:6px 0'><tr>{th}</tr>{corpo}</table>"))

def caixa(texto, tipo="info"):
    """Mostra uma mensagem colorida: tipo = 'ok', 'erro', 'aviso' ou 'info'."""
    cores = {"ok": "#1b7f3b", "erro": "#b3261e", "aviso": "#b36b00", "info": "#1f5fa8"}
    display(HTML(f"<div style='border-left:5px solid {cores[tipo]};padding:6px 12px;margin:6px 0;"
                 f"font-family:monospace;white-space:pre-wrap'>{html.escape(texto)}</div>"))

# RF1: converte minutos desde a meia-noite de volta para "HH:MM" (só para exibir)
def hhmm(minutos):
    return f"{int(minutos) // 60:02d}:{int(minutos) % 60:02d}"

print("✅ Ferramentas de visualização carregadas!")

✅ Ferramentas de visualização carregadas!


## Etapa 2 - Programa de exemplo e gramática
Programa com as regras da estufa. Já usa horário (RF1), SENAO (RF2) e as ações novas `irrigar` e `ventilar` (RF3). O `28 + 2` e o `* 1` estão aí para testar o otimizador.

In [3]:
# 📜 CÉLULA 2 — Um programa em CasaScript 2.0 (cenário: estufa inteligente)

PROGRAMA_EXEMPLO = '''
# Regras da estufa inteligente
QUANDO umidade_solo < 30 E nivel_reservatorio > 10 ENTAO
    irrigar(irrigador, 10)
SENAO
    desligar(irrigador)
FIM

QUANDO temperatura > 28 + 2 ENTAO
    ventilar(ventilador, 80);
    abrir(janela_teto)
SENAO
    desligar(ventilador);
    fechar(janela_teto)
FIM

QUANDO clima == "chuvoso" ENTAO
    fechar(janela_teto);
    notificar("Chuva! Janela do teto fechada.")
FIM

QUANDO relogio >= 18:30 E relogio < 22:00 E luminosidade * 1 < 5000 ENTAO
    ajustar(lampada_cultivo, 80)
SENAO
    desligar(lampada_cultivo)
FIM

QUANDO nivel_reservatorio < 15 ENTAO
    notificar("Reservatório quase vazio!")
FIM
'''
print(PROGRAMA_EXEMPLO)


# Regras da estufa inteligente
QUANDO umidade_solo < 30 E nivel_reservatorio > 10 ENTAO
    irrigar(irrigador, 10)
SENAO
    desligar(irrigador)
FIM

QUANDO temperatura > 28 + 2 ENTAO
    ventilar(ventilador, 80);
    abrir(janela_teto)
SENAO
    desligar(ventilador);
    fechar(janela_teto)
FIM

QUANDO clima == "chuvoso" ENTAO
    fechar(janela_teto);
    notificar("Chuva! Janela do teto fechada.")
FIM

QUANDO relogio >= 18:30 E relogio < 22:00 E luminosidade * 1 < 5000 ENTAO
    ajustar(lampada_cultivo, 80)
SENAO
    desligar(lampada_cultivo)
FIM

QUANDO nivel_reservatorio < 15 ENTAO
    notificar("Reservatório quase vazio!")
FIM



### Gramática
Mudanças:
- `regra` agora tem um `[senao]` opcional (RF2). Como `acoes` precisa de pelo menos uma ação, `SENAO FIM` já dá erro de sintaxe sozinho.
- token novo `HORARIO` (RF1). Colocamos prioridade 2 (`HORARIO.2`) porque senão o `18:30` era lido como NUMERO `18` e depois dava erro no `:` (o mesmo problema de prioridade da Prática 1).
- a validação de 00:00 a 23:59 ficou num callback do lexer (`validar_horario`), porque com regex ficava bem mais difícil de ler.

In [4]:
# 📖 CÉLULA 3 — A gramática da CasaScript 2.0

GRAMATICA = r'''
// ===================== REGRAS SINTÁTICAS =====================
programa   : regra+
regra      : "QUANDO" expr "ENTAO" acoes [senao] "FIM"      // RF2
senao      : "SENAO" acoes                                  // RF2
acoes      : acao (";" acao)* ";"?
acao       : NOME "(" [argumentos] ")"
argumentos : expr ("," expr)*

// Precedência: quanto mais embaixo, mais "forte" (calculado primeiro)
?expr       : ou
?ou         : e          | ou "OU" e             -> op_ou
?e          : nao        | e "E" nao             -> op_e
?nao        : comparacao | "NAO" nao             -> op_nao
?comparacao : soma       | soma OP_COMP soma     -> op_comp
?soma       : produto    | soma OP_SOMA produto  -> op_arit
?produto    : atomo      | produto OP_MULT atomo -> op_arit
?atomo      : HORARIO       -> horario                     // RF1
            | NUMERO        -> numero
            | TEXTO         -> texto
            | "VERDADEIRO"  -> verdadeiro
            | "FALSO"       -> falso
            | NOME          -> variavel
            | "(" expr ")"

// ===================== TERMINAIS LÉXICOS =====================
HORARIO.2  : /\d\d:\d\d/          // RF1 - prioridade maior que NUMERO
OP_COMP    : "==" | "!=" | ">=" | "<=" | ">" | "<"
OP_SOMA    : "+" | "-"
OP_MULT    : "*" | "/"
NUMERO     : /\d+(\.\d+)?/
TEXTO      : /"[^"\n]*"/
NOME       : /[a-z_][a-z0-9_]*/
COMENTARIO : /#[^\n]*/

%import common.WS
%ignore WS
%ignore COMENTARIO
'''

# RF1: erro léxico próprio para horários fora de 00:00..23:59
class ErroHorario(Exception):
    def __init__(self, token):
        self.token, self.line, self.column = token, token.line, token.column
        self.pos_in_stream = token.start_pos
        super().__init__(f"horário inválido: {token}")

def validar_horario(token):
    """Callback chamado pelo LEXER a cada token HORARIO reconhecido."""
    hh, mm = map(int, token.split(":"))
    if hh > 23 or mm > 59:
        raise ErroHorario(token)
    return token

parser = Lark(GRAMATICA, start="programa", parser="lalr", propagate_positions=True,
              lexer_callbacks={"HORARIO": validar_horario})
print("✅ Gramática carregada! O Lark gerou um parser LALR(1) para a CasaScript 2.0.")

✅ Gramática carregada! O Lark gerou um parser LALR(1) para a CasaScript 2.0.


## Etapa 3 - Análise léxica

In [5]:
# 🔤 CÉLULA 4 — FASE 1: Análise Léxica (texto -> tokens)

def analisar_lexico(fonte):
    """Quebra o texto em tokens usando APENAS a parte léxica da gramática."""
    return list(parser.lex(fonte))

tokens = analisar_lexico(PROGRAMA_EXEMPLO)
tabela(["#", "Linha:Col", "Token (categoria)", "Lexema (texto)"],
       [(i, f"{t.line}:{t.column}", t.type, t.value) for i, t in enumerate(tokens, 1)][:20],
       titulo=f"🔤 {len(tokens)} tokens encontrados (mostrando os 20 primeiros)")

# RF1: 18:30 é UM token; 25:00 e 18:75 são erros léxicos
linhas = []
for trecho in ["relogio >= 18:30", "relogio >= 25:00", "relogio >= 18:75"]:
    try:
        linhas.append((trecho, " | ".join(f"{t.type}:{t.value}" for t in analisar_lexico(trecho)), "✅"))
    except ErroHorario as e:
        linhas.append((trecho, f"ERRO LÉXICO: '{e.token}' não é um horário válido", "❌"))
tabela(["Entrada", "Tokens", ""], linhas, titulo="🕒 RF1 — o literal de horário")

#,Linha:Col,Token (categoria),Lexema (texto)
1,3:1,QUANDO,QUANDO
2,3:8,NOME,umidade_solo
3,3:21,OP_COMP,<
4,3:23,NUMERO,30
5,3:26,E,E
6,3:28,NOME,nivel_reservatorio
7,3:47,OP_COMP,>
8,3:49,NUMERO,10
9,3:52,ENTAO,ENTAO
10,4:5,NOME,irrigar


Entrada,Tokens,
relogio >= 18:30,NOME:relogio | OP_COMP:>= | HORARIO:18:30,✅
relogio >= 25:00,ERRO LÉXICO: '25:00' não é um horário válido,❌
relogio >= 18:75,ERRO LÉXICO: '18:75' não é um horário válido,❌


## Etapa 4 - Análise sintática
Mostrando só a Regra #1 para ver o nó `senao` na árvore.

In [6]:
# 🌳 CÉLULA 5 — FASE 2: Análise Sintática (tokens -> árvore sintática)

arvore = parser.parse(PROGRAMA_EXEMPLO)
print("🌳 Árvore sintática da Regra #1:\n")
print(arvore.children[0].pretty())

🌳 Árvore sintática da Regra #1:

regra
  op_e
    op_comp
      variavel	umidade_solo
      <
      numero	30
    op_comp
      variavel	nivel_reservatorio
      >
      numero	10
  acoes
    acao
      irrigar
      argumentos
        variavel	irrigador
        numero	10
  senao
    acoes
      acao
        desligar
        argumentos
          variavel	irrigador



## Etapa 5 - AST
Novidades: o nó `Horario` guarda o horário em minutos (18:30 -> 1110), e a `Regra` agora tem o campo `senao` (fica `None` quando a regra não tem SENAO).
Também passamos a função `mostrar()` para cá, porque ela agora é usada na semântica para comparar condições.

In [7]:
# 🧩 CÉLULA 6 — FASE 2 (parte 2): da árvore sintática para a AST

# ---------- Os "tipos de peça" da AST ----------
@dataclass
class Numero:      valor: float;  linha: int = 0; coluna: int = 0
@dataclass
class Texto:       valor: str;    linha: int = 0; coluna: int = 0
@dataclass
class Logico:      valor: bool;   linha: int = 0; coluna: int = 0
@dataclass
class Horario:     valor: int;    linha: int = 0; coluna: int = 0    # RF1: minutos desde 00:00
@dataclass
class Variavel:    nome: str;     linha: int = 0; coluna: int = 0
@dataclass
class Aritmetica:  op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0
@dataclass
class Comparacao:  op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0
@dataclass
class Logica:      op: str; esq: object; dir: object; linha: int = 0; coluna: int = 0   # "E" ou "OU"
@dataclass
class Nao:         expr: object;  linha: int = 0; coluna: int = 0
@dataclass
class Acao:        nome: str; args: list; linha: int = 0; coluna: int = 0
@dataclass
class Regra:
    numero: int
    condicao: object
    acoes: list
    senao: list = None                                                # RF2: None = sem SENAO
    fonte: str = ""
    linha: int = 0
    coluna: int = 0
@dataclass
class Programa:    regras: list

# ---------- O "tradutor" árvore do Lark -> AST ----------
@v_args(meta=True)
class ConstrutorAST(Transformer):
    def __init__(self, fonte):
        super().__init__()
        self.fonte = fonte

    def numero(self, meta, f):
        t = f[0]
        return Numero(float(t) if "." in t else int(t), meta.line, meta.column)
    def horario(self, meta, f):                                      # RF1
        hh, mm = map(int, str(f[0]).split(":"))
        return Horario(hh * 60 + mm, meta.line, meta.column)         # 18:30 -> 1110
    def texto(self, meta, f):      return Texto(f[0][1:-1], meta.line, meta.column)
    def verdadeiro(self, meta, f): return Logico(True, meta.line, meta.column)
    def falso(self, meta, f):      return Logico(False, meta.line, meta.column)
    def variavel(self, meta, f):   return Variavel(str(f[0]), meta.line, meta.column)
    def op_arit(self, meta, f):    return Aritmetica(str(f[1]), f[0], f[2], meta.line, meta.column)
    def op_comp(self, meta, f):    return Comparacao(str(f[1]), f[0], f[2], meta.line, meta.column)
    def op_e(self, meta, f):       return Logica("E", f[0], f[1], meta.line, meta.column)
    def op_ou(self, meta, f):      return Logica("OU", f[0], f[1], meta.line, meta.column)
    def op_nao(self, meta, f):     return Nao(f[0], meta.line, meta.column)
    def argumentos(self, meta, f): return list(f)
    def acao(self, meta, f):       return Acao(str(f[0]), f[1] or [], meta.line, meta.column)
    def acoes(self, meta, f):      return list(f)
    def senao(self, meta, f):      return f[0]                       # RF2: a lista de ações
    def regra(self, meta, f):
        trecho = self.fonte[meta.start_pos:meta.end_pos]
        return Regra(0, f[0], f[1], f[2], trecho, meta.line, meta.column)   # f[2] = None sem SENAO
    def programa(self, meta, f):
        for i, r in enumerate(f, 1):
            r.numero = i
        return Programa(list(f))

# ---------- Expressão -> texto (usado no relatório do otimizador e na semântica) ----------
def mostrar(no):
    if isinstance(no, Numero):   return str(no.valor)
    if isinstance(no, Texto):    return f'"{no.valor}"'
    if isinstance(no, Logico):   return "VERDADEIRO" if no.valor else "FALSO"
    if isinstance(no, Horario):  return hhmm(no.valor)                # RF1
    if isinstance(no, Variavel): return no.nome
    if isinstance(no, Nao):      return f"NAO {mostrar(no.expr)}"
    return f"({mostrar(no.esq)} {no.op} {mostrar(no.dir)})"

# ---------- Desenhista de AST ----------
def rotulo(no):
    if isinstance(no, Programa):   return "📦 Programa"
    if isinstance(no, Regra):      return f"📜 Regra #{no.numero}"
    if isinstance(no, Numero):     return f"🔢 Numero {no.valor}"
    if isinstance(no, Texto):      return f'💬 Texto "{no.valor}"'
    if isinstance(no, Logico):     return f"🔘 Logico {'VERDADEIRO' if no.valor else 'FALSO'}"
    if isinstance(no, Horario):    return f"🕒 Horario {hhmm(no.valor)} (= {no.valor} min)"
    if isinstance(no, Variavel):   return f"📡 Variavel {no.nome}"
    if isinstance(no, Aritmetica): return f"➕ Aritmetica ({no.op})"
    if isinstance(no, Comparacao): return f"⚖️ Comparacao ({no.op})"
    if isinstance(no, Logica):     return f"🔗 Logica ({no.op})"
    if isinstance(no, Nao):        return "🚫 Nao"
    if isinstance(no, Acao):       return f"⚡ Acao {no.nome}()"
    return str(no)

def filhos(no):
    if isinstance(no, Programa): return [("", r) for r in no.regras]
    if isinstance(no, Regra):
        return ([("SE: ", no.condicao)] + [("FAÇA: ", a) for a in no.acoes]
                + [("SENÃO: ", a) for a in (no.senao or [])])                    # RF2
    if isinstance(no, (Aritmetica, Comparacao, Logica)): return [("", no.esq), ("", no.dir)]
    if isinstance(no, Nao):  return [("", no.expr)]
    if isinstance(no, Acao): return [("", a) for a in no.args]
    return []

def desenhar_ast(no, prefixo="", ultimo=True, raiz=True, etiqueta=""):
    conector = "" if raiz else ("└── " if ultimo else "├── ")
    linhas = [prefixo + conector + etiqueta + rotulo(no)]
    novo_prefixo = prefixo + ("" if raiz else ("    " if ultimo else "│   "))
    lista = filhos(no)
    for i, (et, filho) in enumerate(lista):
        linhas.append(desenhar_ast(filho, novo_prefixo, i == len(lista) - 1, False, et))
    return "\n".join(linhas)

def construir_ast(fonte):
    return ConstrutorAST(fonte).transform(parser.parse(fonte))

ast_exemplo = construir_ast(PROGRAMA_EXEMPLO)
print(desenhar_ast(Programa([ast_exemplo.regras[0], ast_exemplo.regras[3]])))

📦 Programa
├── 📜 Regra #1
│   ├── SE: 🔗 Logica (E)
│   │   ├── ⚖️ Comparacao (<)
│   │   │   ├── 📡 Variavel umidade_solo
│   │   │   └── 🔢 Numero 30
│   │   └── ⚖️ Comparacao (>)
│   │       ├── 📡 Variavel nivel_reservatorio
│   │       └── 🔢 Numero 10
│   ├── FAÇA: ⚡ Acao irrigar()
│   │   ├── 📡 Variavel irrigador
│   │   └── 🔢 Numero 10
│   └── SENÃO: ⚡ Acao desligar()
│       └── 📡 Variavel irrigador
└── 📜 Regra #4
    ├── SE: 🔗 Logica (E)
    │   ├── 🔗 Logica (E)
    │   │   ├── ⚖️ Comparacao (>=)
    │   │   │   ├── 📡 Variavel relogio
    │   │   │   └── 🕒 Horario 18:30 (= 1110 min)
    │   │   └── ⚖️ Comparacao (<)
    │   │       ├── 📡 Variavel relogio
    │   │       └── 🕒 Horario 22:00 (= 1320 min)
    │   └── ⚖️ Comparacao (<)
    │       ├── ➕ Aritmetica (*)
    │       │   ├── 📡 Variavel luminosidade
    │       │   └── 🔢 Numero 1
    │       └── 🔢 Numero 5000
    ├── FAÇA: ⚡ Acao ajustar()
    │   ├── 📡 Variavel lampada_cultivo
    │   └── 🔢 Numero 80
    └── SENÃO: ⚡ Acao

## Etapa 6 - Análise semântica

Tabela de símbolos da estufa:

| Sensor | Tipo | Valores |
|---|---|---|
| umidade_solo | número | 0 a 100 % |
| temperatura | número | -10 a 50 °C |
| luminosidade | número | 0 a 100000 lux |
| nivel_reservatorio | número | 0 a 100 % |
| clima | texto | ensolarado, nublado, chuvoso |
| pragas | lógico | VERDADEIRO/FALSO |
| relogio | horário | 00:00 a 23:59 |

Dispositivos: `irrigador` e `gotejador` (irrigador), `lampada_cultivo` (lampada), `ventilador` (ventilador), `janela_teto` (janela).

Ações novas: `irrigar(dispositivo, minutos)` de 1 a 60 e `ventilar(dispositivo, velocidade)` de 10 a 100.

In [8]:
# 🧠 CÉLULA 7 — FASE 3: Análise Semântica (a frase FAZ SENTIDO?)

# ---------- TABELA DE SÍMBOLOS da estufa (RF3) ----------
SENSORES = {
    "umidade_solo":       "numero",
    "temperatura":        "numero",
    "luminosidade":       "numero",
    "nivel_reservatorio": "numero",
    "clima":              "texto",
    "pragas":             "logico",
    "relogio":            "horario",     # RF1: sensor do tipo horário
}
FAIXA_SENSOR = {"umidade_solo": (0, 100), "temperatura": (-10, 50),
                "luminosidade": (0, 100000), "nivel_reservatorio": (0, 100)}
VALORES_TEXTO = {"clima": {"ensolarado", "nublado", "chuvoso"}}

DISPOSITIVOS = {
    "irrigador":       "irrigador",
    "gotejador":       "irrigador",
    "lampada_cultivo": "lampada",
    "ventilador":      "ventilador",
    "janela_teto":     "janela",
}

LIGAVEIS = {"irrigador", "lampada", "ventilador"}
ACOES = {   # nome -> (parâmetros, tipos de aparelho que aceitam a ação)
    "ligar":     (["dispositivo"],           LIGAVEIS),
    "desligar":  (["dispositivo"],           LIGAVEIS),
    "ajustar":   (["dispositivo", "numero"], {"lampada"}),
    "irrigar":   (["dispositivo", "numero"], {"irrigador"}),     # nova: minutos de irrigação
    "ventilar":  (["dispositivo", "numero"], {"ventilador"}),    # nova: velocidade em %
    "abrir":     (["dispositivo"],           {"janela"}),
    "fechar":    (["dispositivo"],           {"janela"}),
    "notificar": (["texto"],                 set()),
}
FAIXA_ACAO = {   # (ação, tipo de aparelho) -> (mínimo, máximo, unidade)
    ("ajustar",  "lampada"):    (0, 100, "% de brilho"),
    ("irrigar",  "irrigador"):  (1, 60,  "minutos"),
    ("ventilar", "ventilador"): (10, 100, "% de velocidade"),
}

class AnalisadorSemantico:
    def __init__(self):
        self.erros, self.avisos = [], []

    def erro(self, no, msg):  self.erros.append(f"❌ Linha {no.linha}, col {no.coluna}: {msg}")
    def aviso(self, no, msg): self.avisos.append(f"⚠️ Linha {no.linha}, col {no.coluna}: {msg}")
    def sugestao(self, nome, conhecidos):
        parecido = difflib.get_close_matches(nome, conhecidos, n=1)
        return f" Você quis dizer '{parecido[0]}'?" if parecido else ""

    # ---------- descobre o TIPO de uma expressão ----------
    def tipo(self, no):
        if isinstance(no, Numero):  return "numero"
        if isinstance(no, Texto):   return "texto"
        if isinstance(no, Logico):  return "logico"
        if isinstance(no, Horario): return "horario"                  # RF1
        if isinstance(no, Variavel):
            if no.nome in SENSORES:
                return SENSORES[no.nome]
            if no.nome in DISPOSITIVOS:
                self.erro(no, f"'{no.nome}' é um DISPOSITIVO, não um sensor. Dispositivos não têm valor para comparar.")
            else:
                self.erro(no, f"sensor '{no.nome}' não existe." + self.sugestao(no.nome, SENSORES))
            return "erro"
        if isinstance(no, Aritmetica):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != "numero" or td != "numero":
                self.erro(no, f"a conta '{no.op}' só funciona com números (recebeu {te} {no.op} {td}).")
                return "erro"
            if no.op == "/" and isinstance(no.dir, Numero) and no.dir.valor == 0:
                self.erro(no, "divisão por zero!")
                return "erro"
            return "numero"
        if isinstance(no, Comparacao):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != td:
                self.erro(no, f"não dá para comparar {te} com {td} usando '{no.op}'.")
                return "erro"
            if te in ("texto", "logico") and no.op not in ("==", "!="):
                self.erro(no, f"'{no.op}' não faz sentido para {te}. Use apenas == ou !=.")
                return "erro"
            self.verificar_valores(no)
            return "logico"
        if isinstance(no, Logica):
            te, td = self.tipo(no.esq), self.tipo(no.dir)
            if "erro" in (te, td): return "erro"
            if te != "logico" or td != "logico":
                self.erro(no, f"'{no.op}' liga duas condições VERDADEIRO/FALSO (recebeu {te} {no.op} {td}).")
                return "erro"
            return "logico"
        if isinstance(no, Nao):
            t = self.tipo(no.expr)
            if t not in ("logico", "erro"):
                self.erro(no, f"NAO só pode negar uma condição (recebeu {t}).")
                return "erro"
            return t
        return "erro"

    # ---------- regras de NEGÓCIO: valores que nunca acontecem ----------
    def verificar_valores(self, comp):
        var, const = comp.esq, comp.dir
        if not isinstance(var, Variavel):
            var, const = comp.dir, comp.esq
        if not isinstance(var, Variavel):
            return
        if isinstance(const, Numero) and var.nome in FAIXA_SENSOR:
            minimo, maximo = FAIXA_SENSOR[var.nome]
            if not (minimo <= const.valor <= maximo):
                self.aviso(comp, f"'{var.nome}' vai de {minimo} a {maximo}; o valor {const.valor} está fora da faixa.")
        if isinstance(const, Texto) and var.nome in VALORES_TEXTO:
            validos = VALORES_TEXTO[var.nome]
            if const.valor not in validos:
                self.aviso(comp, f"'{var.nome}' nunca vale \"{const.valor}\" (valores possíveis: {sorted(validos)})."
                                 + self.sugestao(const.valor, validos))

    # ---------- verifica uma AÇÃO (vale para ENTAO e para SENAO) ----------
    def verificar_acao(self, acao):
        if acao.nome not in ACOES:
            self.erro(acao, f"ação '{acao.nome}' não existe." + self.sugestao(acao.nome, ACOES))
            return
        params, aceita = ACOES[acao.nome]
        if len(acao.args) != len(params):
            self.erro(acao, f"'{acao.nome}' espera {len(params)} argumento(s), mas recebeu {len(acao.args)}.")
            return
        for param, arg in zip(params, acao.args):
            if param == "dispositivo":
                if not isinstance(arg, Variavel) or arg.nome not in DISPOSITIVOS:
                    nome = arg.nome if isinstance(arg, Variavel) else "?"
                    self.erro(arg, f"'{acao.nome}' precisa de um dispositivo; '{nome}' não é um."
                                   + (self.sugestao(nome, DISPOSITIVOS) if nome != "?" else ""))
                    return
                tipo_disp = DISPOSITIVOS[arg.nome]
                if tipo_disp not in aceita:
                    self.erro(acao, f"não é possível '{acao.nome}' um(a) {tipo_disp} ('{arg.nome}').")
                    return
            else:
                t = self.tipo(arg)
                if t != "erro" and t != param:
                    self.erro(arg, f"'{acao.nome}' esperava {param}, mas recebeu {t}.")
                    return
        # Ações (dispositivo, numero): o número está dentro da faixa do aparelho?
        if params == ["dispositivo", "numero"] and isinstance(acao.args[1], Numero):
            faixa = FAIXA_ACAO.get((acao.nome, DISPOSITIVOS[acao.args[0].nome]))
            if faixa and not (faixa[0] <= acao.args[1].valor <= faixa[1]):
                self.erro(acao.args[1], f"'{acao.args[0].nome}' aceita valores de {faixa[0]} a {faixa[1]} ({faixa[2]}) em '{acao.nome}'.")

    # ---------- verifica um BLOCO de ações (ENTAO ou SENAO) ----------
    def verificar_bloco(self, regra, nome_bloco, acoes):
        usados, vistas = {}, {}
        for acao in acoes:
            self.verificar_acao(acao)
            # RF3: a mesma ação, com os mesmos argumentos, repetida no bloco
            assinatura = f"{acao.nome}({', '.join(mostrar(a) for a in acao.args)})"
            if assinatura in vistas:
                self.aviso(acao, f"a ação '{assinatura}' está repetida no bloco {nome_bloco} da Regra #{regra.numero} "
                                 f"(primeira vez na linha {vistas[assinatura][0]}, col {vistas[assinatura][1]}).")
            else:
                vistas[assinatura] = (acao.linha, acao.coluna)
            if acao.nome in ("ligar", "desligar") and acao.args and isinstance(acao.args[0], Variavel):
                usados.setdefault(acao.args[0].nome, set()).add(acao.nome)
        for disp, nomes in usados.items():
            if len(nomes) == 2:
                self.aviso(regra, f"o bloco {nome_bloco} da Regra #{regra.numero} liga E desliga '{disp}' ao mesmo tempo.")

    # ---------- ponto de entrada ----------
    def analisar(self, programa):
        condicoes_vistas = {}
        for regra in programa.regras:
            t = self.tipo(regra.condicao)
            if t not in ("logico", "erro"):
                self.erro(regra.condicao, f"a condição da Regra #{regra.numero} precisa ser VERDADEIRO/FALSO, mas é {t}.")
            self.verificar_bloco(regra, "ENTAO", regra.acoes)
            if regra.senao is not None:                                  # RF2/RF3
                self.verificar_bloco(regra, "SENAO", regra.senao)
            # RF3: duas regras com exatamente a mesma condição
            chave = mostrar(regra.condicao)
            if chave in condicoes_vistas:
                self.aviso(regra, f"a Regra #{regra.numero} tem exatamente a mesma condição da Regra "
                                  f"#{condicoes_vistas[chave]}: {chave}")
            else:
                condicoes_vistas[chave] = regra.numero
        return self.erros, self.avisos

# ---------- Mostrando a tabela de símbolos ----------
tabela(["Nome", "Categoria", "Tipo", "Faixa / valores"],
       [(n, "sensor", t, FAIXA_SENSOR.get(n) or sorted(VALORES_TEXTO.get(n, [])) or
         ("00:00..23:59" if t == "horario" else "VERDADEIRO/FALSO")) for n, t in SENSORES.items()] +
       [(n, "dispositivo", t, ", ".join(a for a, (_, ac) in ACOES.items() if t in ac)) for n, t in DISPOSITIVOS.items()],
       titulo="🌱 Tabela de Símbolos da estufa")

erros, avisos = AnalisadorSemantico().analisar(ast_exemplo)
caixa("Programa de exemplo: nenhum erro semântico! ✅" if not erros else "\n".join(erros),
      "ok" if not erros else "erro")
if avisos: caixa("\n".join(avisos), "aviso")

Nome,Categoria,Tipo,Faixa / valores
umidade_solo,sensor,numero,"(0, 100)"
temperatura,sensor,numero,"(-10, 50)"
luminosidade,sensor,numero,"(0, 100000)"
nivel_reservatorio,sensor,numero,"(0, 100)"
clima,sensor,texto,"['chuvoso', 'ensolarado', 'nublado']"
pragas,sensor,logico,VERDADEIRO/FALSO
relogio,sensor,horario,00:00..23:59
irrigador,dispositivo,irrigador,"ligar, desligar, irrigar"
gotejador,dispositivo,irrigador,"ligar, desligar, irrigar"
lampada_cultivo,dispositivo,lampada,"ligar, desligar, ajustar"


### Testando um programa com vários erros

In [9]:
# 🐛 CÉLULA 8 — Programa com erros e avisos semânticos (léxico e sintaxe estão corretos!)

PROGRAMA_COM_ERROS = '''
QUANDO temperatura > "quente" ENTAO ligar(ventilador) FIM
QUANDO umidade_soloo < 30 ENTAO irrigar(irrigador, 10) FIM
QUANDO relogio >= 18 ENTAO ajustar(lampada_cultivo, 80) FIM
QUANDO clima == "chuvozo" ENTAO fechar(janela_teto) SENAO abrir(janela_teto); irrigar(janela_teto, 5) FIM
QUANDO pragas ENTAO ventilar(ventilador, 150) SENAO desligar(ventilador); regar(irrigador) FIM
QUANDO umidade_solo < 20 ENTAO irrigar(irrigador, 10); irrigar(irrigador, 10) FIM
QUANDO umidade_solo < 20 ENTAO notificar("Solo seco!") FIM
QUANDO relogio + 30 > 06:00 ENTAO ligar(lampada_cultivo) FIM
'''
erros, avisos = AnalisadorSemantico().analisar(construir_ast(PROGRAMA_COM_ERROS))
caixa("\n".join(erros), "erro")
caixa("\n".join(avisos), "aviso")

Os dois avisos novos aparecem no final (ação repetida e condição repetida). Os erros dentro do SENAO (`irrigar(janela_teto, 5)` e `regar(irrigador)`) também foram pegos.

## Etapa 7 - Otimização
Além das 4 técnicas da aula, colocamos mais duas (RF4):
- dupla negação: `NAO NAO x` -> `x`
- identidades: `x + 0`, `x - 0`, `x * 1`, `x / 1` -> `x` e `x * 0` -> `0`

Como estava na dica, primeiro roda a otimização que já existia e depois as novas em cima do resultado.

In [10]:
# 🚀 CÉLULA 9 — FASE 4: Otimização (reescrever a AST sem mudar o significado)

OPS_ARIT = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}
OPS_COMP = {"==": operator.eq, "!=": operator.ne, ">": operator.gt,
            "<": operator.lt, ">=": operator.ge, "<=": operator.le}

def e_constante(no):
    return isinstance(no, (Numero, Texto, Logico, Horario))          # RF1: Horario é constante

def eh_numero(no, valor):
    return isinstance(no, Numero) and no.valor == valor

class Otimizador:
    def __init__(self):
        self.relatorio = []

    def registrar(self, tecnica, antes, depois):
        self.relatorio.append((tecnica, mostrar(antes), mostrar(depois)))

    def otimizar_expr(self, no):
        no = self.tecnicas_originais(no)       # 1) folding, simplificação lógica (como na aula)
        no = self.tecnicas_novas(no)           # 2) RF4 - técnicas novas
        return no

    # ---------- técnicas originais ----------
    def tecnicas_originais(self, no):
        # Otimiza primeiro os filhos (de baixo para cima)
        if isinstance(no, (Aritmetica, Comparacao, Logica)):
            no = type(no)(no.op, self.otimizar_expr(no.esq), self.otimizar_expr(no.dir), no.linha, no.coluna)
        elif isinstance(no, Nao):
            no = Nao(self.otimizar_expr(no.expr), no.linha, no.coluna)

        # CONSTANT FOLDING
        if isinstance(no, Aritmetica) and e_constante(no.esq) and e_constante(no.dir):
            valor = OPS_ARIT[no.op](no.esq.valor, no.dir.valor)
            if isinstance(valor, float) and valor.is_integer():
                valor = int(valor)
            novo = Numero(valor, no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo
        if isinstance(no, Comparacao) and e_constante(no.esq) and e_constante(no.dir):
            novo = Logico(OPS_COMP[no.op](no.esq.valor, no.dir.valor), no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo
        if isinstance(no, Nao) and isinstance(no.expr, Logico):
            novo = Logico(not no.expr.valor, no.linha, no.coluna)
            self.registrar("Constant folding", no, novo)
            return novo

        # SIMPLIFICAÇÃO LÓGICA
        if isinstance(no, Logica):
            for const, outro in ((no.esq, no.dir), (no.dir, no.esq)):
                if isinstance(const, Logico):
                    if no.op == "E":
                        novo = outro if const.valor else Logico(False, no.linha, no.coluna)
                    else:
                        novo = Logico(True, no.linha, no.coluna) if const.valor else outro
                    self.registrar("Simplificação lógica", no, novo)
                    return novo
        return no

    # ---------- RF4: técnicas novas ----------
    def tecnicas_novas(self, no):
        # 5) ELIMINAÇÃO DE DUPLA NEGAÇÃO: NAO NAO x -> x
        if isinstance(no, Nao) and isinstance(no.expr, Nao):
            novo = no.expr.expr
            self.registrar("Dupla negação", no, novo)
            return novo

        # 6) IDENTIDADES ARITMÉTICAS
        if isinstance(no, Aritmetica):
            e, d, op = no.esq, no.dir, no.op
            novo = None
            if op == "*" and (eh_numero(e, 0) or eh_numero(d, 0)):
                novo = Numero(0, no.linha, no.coluna)                   # x * 0 -> 0
            elif op in ("+", "-") and eh_numero(d, 0):  novo = e        # x + 0, x - 0 -> x
            elif op == "+" and eh_numero(e, 0):         novo = d        # 0 + x -> x
            elif op in ("*", "/") and eh_numero(d, 1):  novo = e        # x * 1, x / 1 -> x
            elif op == "*" and eh_numero(e, 1):         novo = d        # 1 * x -> x
            if novo is not None:
                self.registrar("Identidade aritmética", no, novo)
                return novo
        return no

    def otimizar(self, programa):
        regras = []
        for r in programa.regras:
            cond = self.otimizar_expr(r.condicao)
            # 4) ELIMINAÇÃO DE CÓDIGO MORTO: condição sempre FALSA nunca muda de estado,
            #    então nem o ENTAO nem o SENAO jamais disparam.
            if isinstance(cond, Logico) and not cond.valor:
                self.relatorio.append(("Código morto", f"Regra #{r.numero}", "removida (nunca dispara)"))
                continue
            regras.append(Regra(r.numero, cond, r.acoes, r.senao,
                                r.fonte, r.linha, r.coluna))
        return Programa(regras)

# ---------- Demonstração ----------
PROGRAMA_OTIMIZAVEL = '''
QUANDO NAO NAO pragas ENTAO ventilar(ventilador, 100) FIM
QUANDO umidade_solo * 1 + 0 < 30 E VERDADEIRO ENTAO irrigar(irrigador, 10) FIM
QUANDO temperatura * 0 > 10 ENTAO ligar(lampada_cultivo) FIM
QUANDO NAO NAO NAO (luminosidade / 1 < 1000) ENTAO desligar(lampada_cultivo) FIM
'''
otm = Otimizador()
ast_otimizada = otm.otimizar(construir_ast(PROGRAMA_OTIMIZAVEL))
tabela(["Técnica", "Antes", "Depois"], otm.relatorio, titulo="🚀 Relatório do otimizador")
print(desenhar_ast(ast_otimizada))

Técnica,Antes,Depois
Dupla negação,NAO NAO pragas,pragas
Identidade aritmética,(umidade_solo * 1),umidade_solo
Identidade aritmética,(umidade_solo + 0),umidade_solo
Simplificação lógica,((umidade_solo < 30) E VERDADEIRO),(umidade_solo < 30)
Identidade aritmética,(temperatura * 0),0
Constant folding,(0 > 10),FALSO
Código morto,Regra #3,removida (nunca dispara)
Identidade aritmética,(luminosidade / 1),luminosidade
Dupla negação,NAO NAO (luminosidade < 1000),(luminosidade < 1000)


📦 Programa
├── 📜 Regra #1
│   ├── SE: 📡 Variavel pragas
│   └── FAÇA: ⚡ Acao ventilar()
│       ├── 📡 Variavel ventilador
│       └── 🔢 Numero 100
├── 📜 Regra #2
│   ├── SE: ⚖️ Comparacao (<)
│   │   ├── 📡 Variavel umidade_solo
│   │   └── 🔢 Numero 30
│   └── FAÇA: ⚡ Acao irrigar()
│       ├── 📡 Variavel irrigador
│       └── 🔢 Numero 10
└── 📜 Regra #4
    ├── SE: 🚫 Nao
    │   └── ⚖️ Comparacao (<)
    │       ├── 📡 Variavel luminosidade
    │       └── 🔢 Numero 1000
    └── FAÇA: ⚡ Acao desligar()
        └── 📡 Variavel lampada_cultivo


Na regra 3, o `temperatura * 0` virou 0, e aí `0 > 10` virou FALSO e a regra foi removida como código morto.
Obs: o `x * 0 -> 0` só é seguro porque a semântica já garantiu que x é número.

## Etapa 8 - Geração de código
Cada regra agora tem 3 blocos: condição, ENTAO e SENAO (RF5). Quando não tem SENAO, o bloco fica só com HALT.
O horário vira `PUSH_CONST` com os minutos. Colocamos um comentário no disassembler para mostrar o horário original.

In [11]:
# ⚙️ CÉLULA 10 — FASE 5: Geração de Código (AST -> BYTECODE)

@dataclass
class RegraCompilada:
    numero: int
    fonte: str
    condicao: list      # "a regra deve disparar?"
    acoes: list         # bloco ENTAO (borda de subida)
    senao: list         # RF5: bloco SENAO (borda de descida)

class GeradorDeCodigo:
    def __init__(self):
        self.codigo = []

    def emitir(self, op, arg=None, comentario=""):
        self.codigo.append([op, arg, comentario])
        return len(self.codigo) - 1

    def gerar_expr(self, no):
        if isinstance(no, Horario):                                       # RF1/RF5
            self.emitir("PUSH_CONST", no.valor, f"# {hhmm(no.valor)}")
        elif isinstance(no, (Numero, Texto, Logico)):
            self.emitir("PUSH_CONST", no.valor)
        elif isinstance(no, Variavel):
            self.emitir("LOAD_SENSOR", no.nome)
        elif isinstance(no, Aritmetica):
            self.gerar_expr(no.esq); self.gerar_expr(no.dir)
            self.emitir("BIN_OP", no.op)
        elif isinstance(no, Comparacao):
            self.gerar_expr(no.esq); self.gerar_expr(no.dir)
            self.emitir("COMPARE", no.op)
        elif isinstance(no, Nao):
            self.gerar_expr(no.expr)
            self.emitir("NOT")
        elif isinstance(no, Logica):
            # CURTO-CIRCUITO + backpatching
            self.gerar_expr(no.esq)
            salto = self.emitir("JUMP_IF_FALSE_OR_POP" if no.op == "E" else "JUMP_IF_TRUE_OR_POP", "?")
            self.gerar_expr(no.dir)
            self.codigo[salto][1] = len(self.codigo)

    def gerar_condicao(self, expr):
        self.codigo = []
        self.gerar_expr(expr)
        self.emitir("RETURN")
        return self.codigo

    def gerar_acoes(self, acoes):
        self.codigo = []
        for acao in (acoes or []):                      # RF2: SENAO ausente -> só HALT
            params = ACOES[acao.nome][0]
            for param, arg in zip(params, acao.args):
                if param == "dispositivo":
                    self.emitir("LOAD_DEVICE", arg.nome)
                else:
                    self.gerar_expr(arg)
            self.emitir("CALL_ACTION", (acao.nome, len(acao.args)))
        self.emitir("HALT")
        return self.codigo

    def gerar(self, programa):
        return [RegraCompilada(r.numero, r.fonte, self.gerar_condicao(r.condicao),
                               self.gerar_acoes(r.acoes), self.gerar_acoes(r.senao))
                for r in programa.regras]

def desmontar(codigo, titulo=""):
    """'Disassembler': mostra o bytecode de forma legível (parecido com o módulo dis do Python)."""
    linhas = [titulo] if titulo else []
    for endereco, (op, arg, comentario) in enumerate(codigo):
        if arg is None:                   txt = ""
        elif op == "CALL_ACTION":         txt = f"{arg[0]}  (args={arg[1]})"
        elif op.startswith("JUMP"):       txt = f"-> {arg}"
        elif op in ("BIN_OP", "COMPARE"): txt = arg
        else:                             txt = repr(arg)
        linhas.append(f"  {endereco:>3}  {op:<22}{txt:<14}{comentario}".rstrip())
    return "\n".join(linhas)

def mostrar_bytecode(regras_compiladas):
    for r in regras_compiladas:
        print(f"══════ Regra #{r.numero} ══════")
        print(desmontar(r.condicao, "  [condição]"))
        print(desmontar(r.acoes, "  [ENTAO]"))
        print(desmontar(r.senao, "  [SENAO]"))
        print()

bytecode_exemplo = GeradorDeCodigo().gerar(Otimizador().otimizar(ast_exemplo))
mostrar_bytecode([bytecode_exemplo[0], bytecode_exemplo[3]])

══════ Regra #1 ══════
  [condição]
    0  LOAD_SENSOR           'umidade_solo'
    1  PUSH_CONST            30
    2  COMPARE               <
    3  JUMP_IF_FALSE_OR_POP  -> 7
    4  LOAD_SENSOR           'nivel_reservatorio'
    5  PUSH_CONST            10
    6  COMPARE               >
    7  RETURN
  [ENTAO]
    0  LOAD_DEVICE           'irrigador'
    1  PUSH_CONST            10
    2  CALL_ACTION           irrigar  (args=2)
    3  HALT
  [SENAO]
    0  LOAD_DEVICE           'irrigador'
    1  CALL_ACTION           desligar  (args=1)
    2  HALT

══════ Regra #4 ══════
  [condição]
    0  LOAD_SENSOR           'relogio'
    1  PUSH_CONST            1110          # 18:30
    2  COMPARE               >=
    3  JUMP_IF_FALSE_OR_POP  -> 7
    4  LOAD_SENSOR           'relogio'
    5  PUSH_CONST            1320          # 22:00
    6  COMPARE               <
    7  JUMP_IF_FALSE_OR_POP  -> 11
    8  LOAD_SENSOR           'luminosidade'
    9  PUSH_CONST            5000
   10  COMPARE  

## Etapa 9 - Máquina virtual
Trocamos a classe `Casa` pela `Estufa`, com os dispositivos novos e as ações `irrigar` e `ventilar`. A VM ficou praticamente igual.

In [12]:
# 🖥️ CÉLULA 11 — FASE 6: A Máquina Virtual (executa o bytecode)

class Estufa:
    """A estufa SIMULADA: guarda os sensores e o estado de cada dispositivo."""
    def __init__(self):
        self.sensores = {"umidade_solo": 55, "temperatura": 24, "luminosidade": 20000,
                         "nivel_reservatorio": 80, "clima": "ensolarado", "pragas": False,
                         "relogio": 6 * 60}                                   # relógio em minutos
        self.dispositivos = {
            "irrigador":       {"ligado": False, "minutos": 0},
            "gotejador":       {"ligado": False, "minutos": 0},
            "lampada_cultivo": {"ligado": False, "nivel": 100},
            "ventilador":      {"ligado": False, "velocidade": 50},
            "janela_teto":     {"aberta": False},
        }
        self.log = []

    def executar_acao(self, nome, args):
        agora = f"[{hhmm(self.sensores['relogio'])}]"
        if nome in ("ligar", "desligar"):
            self.dispositivos[args[0]]["ligado"] = (nome == "ligar")
        elif nome == "ajustar":
            self.dispositivos[args[0]].update(ligado=True, nivel=args[1])
        elif nome == "irrigar":                                       # nova
            self.dispositivos[args[0]].update(ligado=True, minutos=args[1])
        elif nome == "ventilar":                                      # nova
            self.dispositivos[args[0]].update(ligado=True, velocidade=args[1])
        elif nome in ("abrir", "fechar"):
            self.dispositivos[args[0]]["aberta"] = (nome == "abrir")
        if nome == "notificar":
            self.log.append(f"{agora} 📱 NOTIFICAÇÃO: {args[0]}")
        else:
            extra = f" -> {args[1]}" if len(args) > 1 else ""
            self.log.append(f"{agora} ⚡ {nome}({args[0]}){extra}")

    def painel_html(self):
        d = self.dispositivos
        def lig(x): return d[x]["ligado"]
        itens = [
            ("💧" if lig("irrigador") else "➖", "Irrigador", f"irrigando {d['irrigador']['minutos']} min" if lig("irrigador") else "parado"),
            ("💧" if lig("gotejador") else "➖", "Gotejador", f"irrigando {d['gotejador']['minutos']} min" if lig("gotejador") else "parado"),
            ("💡" if lig("lampada_cultivo") else "⚫", "Lâmpada de cultivo", f"{d['lampada_cultivo']['nivel']}%" if lig("lampada_cultivo") else "desligada"),
            ("🌀" if lig("ventilador") else "⭕", "Ventilador", f"{d['ventilador']['velocidade']}%" if lig("ventilador") else "desligado"),
            ("🪟" if d["janela_teto"]["aberta"] else "🔒", "Janela do teto", "aberta" if d["janela_teto"]["aberta"] else "fechada"),
        ]
        cards = "".join(f"<div style='display:inline-block;width:140px;margin:4px;padding:8px;border:1px solid #999;"
                        f"border-radius:10px;text-align:center;vertical-align:top'><div style='font-size:28px'>{e}</div>"
                        f"<b>{n}</b><br><small>{s}</small></div>" for e, n, s in itens)
        return f"<div>{cards}</div>"

class MaquinaVirtual:
    def __init__(self, ambiente):
        self.ambiente = ambiente

    def executar(self, codigo, rastrear=False):
        pilha, pc, rastro = [], 0, []
        while True:
            op, arg = codigo[pc][0], codigo[pc][1]
            endereco = pc
            pc += 1

            if   op == "PUSH_CONST":  pilha.append(arg)
            elif op == "LOAD_SENSOR": pilha.append(self.ambiente.sensores[arg])
            elif op == "LOAD_DEVICE": pilha.append(arg)
            elif op == "BIN_OP":
                b, a = pilha.pop(), pilha.pop()
                pilha.append(OPS_ARIT[arg](a, b))
            elif op == "COMPARE":
                b, a = pilha.pop(), pilha.pop()
                pilha.append(OPS_COMP[arg](a, b))
            elif op == "NOT":
                pilha.append(not pilha.pop())
            elif op == "JUMP_IF_FALSE_OR_POP":
                if not pilha[-1]: pc = arg
                else: pilha.pop()
            elif op == "JUMP_IF_TRUE_OR_POP":
                if pilha[-1]: pc = arg
                else: pilha.pop()
            elif op == "CALL_ACTION":
                nome, n = arg
                args = pilha[-n:]
                del pilha[-n:]
                self.ambiente.executar_acao(nome, args)
            elif op == "RETURN":
                if rastrear: rastro.append((endereco, op, "", list(pilha)))
                return pilha.pop(), rastro
            elif op == "HALT":
                if rastrear: rastro.append((endereco, op, "", list(pilha)))
                return None, rastro
            else:
                raise RuntimeError(f"Instrução desconhecida: {op}")

            if rastrear:
                rastro.append((endereco, op, "" if arg is None else arg, list(pilha)))

# ---------- Execução passo a passo da condição da Regra #4 (horários!) ----------
estufa = Estufa()
estufa.sensores.update(relogio=19 * 60, luminosidade=3000)
vm = MaquinaVirtual(estufa)
resultado, rastro = vm.executar(bytecode_exemplo[3].condicao, rastrear=True)
tabela(["Endereço", "Instrução", "Argumento", "Pilha DEPOIS"], rastro,
       titulo=f"🖥️ Regra #4 com relogio=19:00 e luminosidade=3000 -> resultado: {resultado}")

estufa.sensores.update(relogio=7 * 60)
resultado, rastro = vm.executar(bytecode_exemplo[3].condicao, rastrear=True)
tabela(["Endereço", "Instrução", "Argumento", "Pilha DEPOIS"], rastro,
       titulo=f"🖥️ Agora com relogio=07:00 -> resultado: {resultado} (curto-circuito: o resto nem foi lido!)")

Endereço,Instrução,Argumento,Pilha DEPOIS
0,LOAD_SENSOR,relogio,[1140]
1,PUSH_CONST,1110,"[1140, 1110]"
2,COMPARE,>=,[True]
3,JUMP_IF_FALSE_OR_POP,7,[]
4,LOAD_SENSOR,relogio,[1140]
5,PUSH_CONST,1320,"[1140, 1320]"
6,COMPARE,<,[True]
7,JUMP_IF_FALSE_OR_POP,11,[]
8,LOAD_SENSOR,luminosidade,[3000]
9,PUSH_CONST,5000,"[3000, 5000]"


Endereço,Instrução,Argumento,Pilha DEPOIS
0,LOAD_SENSOR,relogio,[420]
1,PUSH_CONST,1110,"[420, 1110]"
2,COMPARE,>=,[False]
3,JUMP_IF_FALSE_OR_POP,7,[False]
7,JUMP_IF_FALSE_OR_POP,11,[False]
11,RETURN,,[False]


## Etapa 10 - Compilador completo

In [13]:
# 🏭 CÉLULA 12 — O COMPILADOR COMPLETO: todas as fases em sequência

NOMES_AMIGAVEIS = {
    "QUANDO": "QUANDO", "ENTAO": "ENTAO", "SENAO": "SENAO", "FIM": "FIM", "E": "E", "OU": "OU", "NAO": "NAO",
    "LPAR": "(", "RPAR": ")", "SEMICOLON": ";", "COMMA": ",", "NOME": "um nome",
    "NUMERO": "um número", "TEXTO": "um texto entre aspas", "OP_COMP": "um comparador (==, >, ...)",
    "OP_SOMA": "+ ou -", "OP_MULT": "* ou /", "VERDADEIRO": "VERDADEIRO", "FALSO": "FALSO",
    "HORARIO": "um horário (HH:MM)", "$END": "fim do programa",
}

def contexto_manual(fonte, linha, coluna):
    texto = fonte.splitlines()[linha - 1]
    return f"{texto}\n{' ' * (coluna - 1)}^"

def erro_amigavel(e, fonte):
    """Traduz as exceções do Lark (e o ErroHorario) para mensagens que um humano entende."""
    if isinstance(e, ErroHorario):                                          # RF1
        return "léxica", (f"🔤 ERRO LÉXICO na linha {e.line}, coluna {e.column}: '{e.token}' não é um horário "
                          f"válido (use 00:00 a 23:59).\n\n{contexto_manual(fonte, e.line, e.column)}")
    contexto = e.get_context(fonte).rstrip()
    if isinstance(e, UnexpectedCharacters):
        return "léxica", (f"🔤 ERRO LÉXICO na linha {e.line}, coluna {e.column}: "
                          f"o caractere '{fonte[e.pos_in_stream]}' não pertence à CasaScript.\n\n{contexto}")
    if isinstance(e, UnexpectedToken):
        esperados = sorted({NOMES_AMIGAVEIS.get(x, x) for x in e.expected})
        achou = "o fim do programa" if e.token.type == "$END" else f"'{e.token}'"
        return "sintática", (f"🌳 ERRO SINTÁTICO na linha {e.line}, coluna {e.column}: encontrei {achou}, "
                             f"mas esperava: {', '.join(esperados)}\n\n{contexto}")
    if isinstance(e, UnexpectedEOF):
        return "sintática", "🌳 ERRO SINTÁTICO: o programa terminou antes da hora (faltou um FIM?)."
    return "desconhecida", str(e)

@dataclass
class ResultadoCompilacao:
    ok: bool = False
    fase_erro: str = None
    mensagens: list = field(default_factory=list)
    avisos: list = field(default_factory=list)
    tokens: list = field(default_factory=list)
    arvore: object = None
    ast: object = None
    ast_otimizada: object = None
    relatorio_otimizacao: list = field(default_factory=list)
    bytecode: list = field(default_factory=list)

def compilar(fonte, otimizar=True):
    r = ResultadoCompilacao()
    try:
        r.tokens = analisar_lexico(fonte)                     # FASE 1 — léxica (inclui validar_horario)
        r.arvore = parser.parse(fonte)                        # FASE 2 — sintática
    except (UnexpectedCharacters, UnexpectedToken, UnexpectedEOF, ErroHorario) as e:
        r.fase_erro, msg = erro_amigavel(e, fonte)
        r.mensagens.append(msg)
        return r
    r.ast = ConstrutorAST(fonte).transform(r.arvore)          # FASE 2b — AST
    erros, r.avisos = AnalisadorSemantico().analisar(r.ast)   # FASE 3 — semântica
    if erros:
        r.fase_erro, r.mensagens = "semântica", erros
        return r
    otm = Otimizador()                                        # FASE 4 — otimização
    r.ast_otimizada = otm.otimizar(r.ast) if otimizar else r.ast
    r.relatorio_otimizacao = otm.relatorio
    r.bytecode = GeradorDeCodigo().gerar(r.ast_otimizada)     # FASE 5 — geração de código
    r.ok = True
    return r

def contar_instrucoes(bytecode):
    return sum(len(b.condicao) + len(b.acoes) + len(b.senao) for b in bytecode)

testes = {
    "✅ Programa correto":   PROGRAMA_EXEMPLO,
    "🔤 Horário inválido":   'QUANDO relogio >= 25:00 ENTAO ligar(lampada_cultivo) FIM',
    "🌳 SENAO vazio":        'QUANDO pragas ENTAO ligar(ventilador) SENAO FIM',
    "🧠 Erro no SENAO":      'QUANDO pragas ENTAO ligar(ventilador) SENAO fechar(ventilador) FIM',
}
for nome, fonte in testes.items():
    res = compilar(fonte)
    print(f"\n━━━━━━━━ {nome} ━━━━━━━━")
    if res.ok:
        caixa(f"Compilado com sucesso! {len(res.tokens)} tokens -> {len(res.bytecode)} regras -> "
              f"{contar_instrucoes(res.bytecode)} instruções de bytecode.", "ok")
    else:
        caixa("\n".join(res.mensagens), "erro")

sem, com = compilar(PROGRAMA_EXEMPLO, otimizar=False), compilar(PROGRAMA_EXEMPLO)
caixa(f"Efeito da otimização no programa da estufa: {contar_instrucoes(sem.bytecode)} instruções sem otimizar, "
      f"{contar_instrucoes(com.bytecode)} otimizado.", "info")


━━━━━━━━ ✅ Programa correto ━━━━━━━━



━━━━━━━━ 🔤 Horário inválido ━━━━━━━━



━━━━━━━━ 🌳 SENAO vazio ━━━━━━━━



━━━━━━━━ 🧠 Erro no SENAO ━━━━━━━━


## Etapa 11 - Central de automação
O ENTAO continua disparando quando a condição vai de FALSO para VERDADEIRO (borda de subida). O SENAO dispara no contrário, quando vai de VERDADEIRO para FALSO (borda de descida).

In [14]:
# 🏠 CÉLULA 13 — A CENTRAL DE AUTOMAÇÃO (o "runtime" da CasaScript 2.0)

class CentralDeAutomacao:
    """Recebe eventos dos sensores e dispara as regras compiladas.
    ENTAO dispara na borda de SUBIDA (FALSO -> VERDADEIRO).
    SENAO dispara na borda de DESCIDA (VERDADEIRO -> FALSO)."""

    def __init__(self, bytecode, ambiente):
        self.regras = bytecode
        self.ambiente = ambiente
        self.vm = MaquinaVirtual(ambiente)
        self.estado_anterior = {r.numero: False for r in bytecode}

    def avaliar_todas(self):
        disparadas = []
        for regra in self.regras:
            agora, _ = self.vm.executar(regra.condicao)
            antes = self.estado_anterior[regra.numero]
            if agora and not antes:                              # borda de subida
                self.vm.executar(regra.acoes)
                disparadas.append(f"#{regra.numero} ENTAO")
            elif antes and not agora:                            # RF5: borda de descida
                if len(regra.senao) > 1:                         # SENAO vazio é só um HALT
                    self.vm.executar(regra.senao)
                    disparadas.append(f"#{regra.numero} SENAO")
            self.estado_anterior[regra.numero] = agora
        return disparadas

    def evento(self, sensor, valor):
        self.ambiente.sensores[sensor] = valor
        return self.avaliar_todas()

# simulação de um dia (RF6) - 12 eventos
UM_DIA = [
    ("relogio", 6 * 60),            # 06:00
    ("umidade_solo", 25),           # solo seco -> irrigar
    ("temperatura", 33),            # calor -> ventilar e abrir a janela
    ("umidade_solo", 60),           # solo molhado -> SENAO: desliga o irrigador
    ("clima", "chuvoso"),           # chuva -> fecha a janela e notifica
    ("temperatura", 26),            # esfriou -> SENAO: desliga o ventilador
    ("clima", "nublado"),
    ("relogio", 18 * 60 + 30),      # 18:30
    ("luminosidade", 3000),         # escureceu -> liga a lâmpada de cultivo
    ("nivel_reservatorio", 12),     # reservatório baixo -> notifica
    ("relogio", 22 * 60),           # 22:00 -> SENAO: desliga a lâmpada
    ("nivel_reservatorio", 90),     # reservatório cheio de novo
]

def mostrar_valor(sensor, valor):
    return hhmm(valor) if SENSORES.get(sensor) == "horario" else valor

def simular(eventos, programa=PROGRAMA_EXEMPLO):
    estufa = Estufa()
    central = CentralDeAutomacao(compilar(programa).bytecode, estufa)
    linhas = []
    for sensor, valor in eventos:
        antes = len(estufa.log)
        disparadas = central.evento(sensor, valor)
        linhas.append((f"{sensor} = {mostrar_valor(sensor, valor)}", ", ".join(disparadas) or "—",
                       " | ".join(estufa.log[antes:]) or "—"))
    return estufa, linhas

estufa_dia, linhas_dia = simular(UM_DIA)
tabela(["Evento", "Regras disparadas", "O que aconteceu"], linhas_dia, titulo="🌱 Simulação de um dia na estufa")
display(HTML(estufa_dia.painel_html()))

Evento,Regras disparadas,O que aconteceu
relogio = 06:00,—,—
umidade_solo = 25,#1 ENTAO,[06:00] ⚡ irrigar(irrigador) -> 10
temperatura = 33,#2 ENTAO,[06:00] ⚡ ventilar(ventilador) -> 80 | [06:00] ⚡ abrir(janela_teto)
umidade_solo = 60,#1 SENAO,[06:00] ⚡ desligar(irrigador)
clima = chuvoso,#3 ENTAO,[06:00] ⚡ fechar(janela_teto) | [06:00] 📱 NOTIFICAÇÃO: Chuva! Janela do teto fechada.
temperatura = 26,#2 SENAO,[06:00] ⚡ desligar(ventilador) | [06:00] ⚡ fechar(janela_teto)
clima = nublado,—,—
relogio = 18:30,—,—
luminosidade = 3000,#4 ENTAO,[18:30] ⚡ ajustar(lampada_cultivo) -> 80
nivel_reservatorio = 12,#5 ENTAO,[18:30] 📱 NOTIFICAÇÃO: Reservatório quase vazio!


Na simulação dá para ver o irrigador ligando de manhã e desligando pelo SENAO quando o solo molha, e a lâmpada acendendo às 18:30 e apagando às 22:00.

## Etapa 12 - Compilar uma vez x interpretar sempre

In [15]:
# ⏱️ CÉLULA 14 — Compilar uma vez x Interpretar sempre

def avaliar_ast(no, sensores):
    """INTERPRETADOR DE ÁRVORE: percorre a AST recursivamente, sem gerar bytecode."""
    if isinstance(no, (Numero, Texto, Logico, Horario)): return no.valor
    if isinstance(no, Variavel):   return sensores[no.nome]
    if isinstance(no, Aritmetica): return OPS_ARIT[no.op](avaliar_ast(no.esq, sensores), avaliar_ast(no.dir, sensores))
    if isinstance(no, Comparacao): return OPS_COMP[no.op](avaliar_ast(no.esq, sensores), avaliar_ast(no.dir, sensores))
    if isinstance(no, Nao):        return not avaliar_ast(no.expr, sensores)
    if isinstance(no, Logica):
        esq = avaliar_ast(no.esq, sensores)
        if no.op == "E":  return esq and avaliar_ast(no.dir, sensores)
        return esq or avaliar_ast(no.dir, sensores)

N = 1000
estufa_teste = Estufa()
res = compilar(PROGRAMA_EXEMPLO)
vm_teste = MaquinaVirtual(estufa_teste)

def estrategia_1_texto():
    for _ in range(N):
        ast = compilar(PROGRAMA_EXEMPLO).ast_otimizada
        for r in ast.regras: avaliar_ast(r.condicao, estufa_teste.sensores)

def estrategia_2_arvore():
    for _ in range(N):
        for r in res.ast_otimizada.regras: avaliar_ast(r.condicao, estufa_teste.sensores)

def estrategia_3_bytecode():
    for _ in range(N):
        for r in res.bytecode: vm_teste.executar(r.condicao)

tempos = []
for nome, funcao in [("1) Reanalisar o texto a cada evento", estrategia_1_texto),
                     ("2) AST pronta + interpretador de árvore", estrategia_2_arvore),
                     ("3) Bytecode pronto + máquina virtual", estrategia_3_bytecode)]:
    inicio = time.perf_counter()
    funcao()
    tempos.append((nome, (time.perf_counter() - inicio) * 1000))

base = tempos[0][1]
tabela(["Estratégia", f"Tempo para {N} eventos", "Comparado à estratégia 1"],
       [(n, f"{t:.1f} ms", f"{base / t:.0f}x mais rápido" if t < base else "—") for n, t in tempos],
       titulo="🏁 Quem vence a corrida?")

Estratégia,Tempo para 1000 eventos,Comparado à estratégia 1
1) Reanalisar o texto a cada evento,2616.7 ms,—
2) AST pronta + interpretador de árvore,4.1 ms,638x mais rápido
3) Bytecode pronto + máquina virtual,4.3 ms,603x mais rápido


## Etapa 13 - CasaScript Studio (adaptado para a estufa)

In [16]:
# 🖱️ CÉLULA 15 — PAINEL COMPLETO: IDE da CasaScript + Estufa Simulada

def criar_abas(conteudos, titulos):
    abas = widgets.Tab(children=conteudos)
    for i, t in enumerate(titulos):
        abas.set_title(i, t)
    return abas

editor = widgets.Textarea(value=PROGRAMA_EXEMPLO.strip(), layout=widgets.Layout(width="100%", height="300px"))
chk_otimizar = widgets.Checkbox(value=True, description="Otimizar")
btn_compilar = widgets.Button(description="▶️ Compilar", button_style="primary")
saida_status = widgets.Output()
saidas = [widgets.Output() for _ in range(6)]
abas_fases = criar_abas(saidas, ["🔤 Tokens", "🌳 Árvore", "🧩 AST", "🧠 Semântica", "🚀 Otimização", "⚙️ Bytecode"])

opcoes_relogio = [(hhmm(m), m) for m in range(0, 24 * 60, 30)]
s_relogio = widgets.SelectionSlider(options=opcoes_relogio, value=6 * 60, description="relogio")
s_umid = widgets.IntSlider(value=55, min=0, max=100, description="umidade_solo")
s_temp = widgets.IntSlider(value=24, min=-10, max=50, description="temperatura")
s_luz = widgets.IntSlider(value=20000, min=0, max=100000, step=500, description="luminosidade")
s_res = widgets.IntSlider(value=80, min=0, max=100, description="reservatório")
s_clima = widgets.Dropdown(options=["ensolarado", "nublado", "chuvoso"], value="ensolarado", description="clima")
s_pragas = widgets.Checkbox(value=False, description="pragas")
controles = {"relogio": s_relogio, "umidade_solo": s_umid, "temperatura": s_temp, "luminosidade": s_luz,
             "nivel_reservatorio": s_res, "clima": s_clima, "pragas": s_pragas}
btn_dia = widgets.Button(description="☀️ Simular um dia", button_style="info")
btn_reset = widgets.Button(description="🔄 Reiniciar estufa")
painel = widgets.HTML()
diario = widgets.HTML()

estado = {"bytecode": None, "central": None}

def atualizar_estufa():
    if estado["central"] is None:
        painel.value = "<i>Compile um programa sem erros para ligar a estufa.</i>"
        diario.value = ""
        return
    estufa = estado["central"].ambiente
    painel.value = estufa.painel_html()
    ultimas = estufa.log[-12:] or ["(nenhuma ação ainda)"]
    diario.value = ("<b>📒 Diário da estufa</b><pre style='margin:4px 0'>" +
                    html.escape("\n".join(ultimas)) + "</pre>")

def reiniciar_estufa(_=None):
    if estado["bytecode"] is None:
        estado["central"] = None
    else:
        estufa = Estufa()
        estufa.sensores.update({nome: w.value for nome, w in controles.items()})
        estado["central"] = CentralDeAutomacao(estado["bytecode"], estufa)
        estado["central"].avaliar_todas()
    atualizar_estufa()

def ao_mudar_sensor(mudanca):
    if estado["central"] is not None:
        nome = [n for n, w in controles.items() if w is mudanca["owner"]][0]
        estado["central"].evento(nome, mudanca["new"])
        atualizar_estufa()

for w in controles.values():
    w.observe(ao_mudar_sensor, names="value")

def simular_dia(_):
    if estado["central"] is None:
        return
    reiniciar_estufa()
    for sensor, valor in UM_DIA:
        controles[sensor].value = valor
        time.sleep(0.4)

def ao_compilar(_):
    for s in saidas + [saida_status]:
        s.clear_output()
    res = compilar(editor.value, otimizar=chk_otimizar.value)
    with saida_status:
        if res.ok:
            caixa(f"Compilado! {len(res.bytecode)} regra(s), {contar_instrucoes(res.bytecode)} instruções. "
                  f"Vá para a aba 🌱 Estufa.", "ok")
        else:
            caixa(f"Falha na fase {res.fase_erro.upper()}:\n" + "\n".join(res.mensagens), "erro")
        if res.avisos:
            caixa("\n".join(res.avisos), "aviso")
    with saidas[0]:
        if res.tokens:
            tabela(["Linha:Col", "Token", "Lexema"], [(f"{t.line}:{t.column}", t.type, t.value) for t in res.tokens])
    with saidas[1]:
        if res.arvore is not None: print(res.arvore.pretty())
    with saidas[2]:
        if res.ast is not None: print(desenhar_ast(res.ast))
    with saidas[3]:
        if res.fase_erro == "semântica": caixa("\n".join(res.mensagens), "erro")
        elif res.ast is not None:        caixa("Nenhum erro semântico. ✅", "ok")
        if res.avisos:                   caixa("\n".join(res.avisos), "aviso")
    with saidas[4]:
        if res.relatorio_otimizacao:
            tabela(["Técnica", "Antes", "Depois"], res.relatorio_otimizacao)
            print(desenhar_ast(res.ast_otimizada))
        elif res.ok:
            print("Nenhuma otimização aplicada.")
    with saidas[5]:
        mostrar_bytecode(res.bytecode)
    estado["bytecode"] = res.bytecode if res.ok else None
    reiniciar_estufa()

btn_compilar.on_click(ao_compilar)
btn_dia.on_click(simular_dia)
btn_reset.on_click(reiniciar_estufa)

aba_compilador = widgets.VBox([editor, widgets.HBox([btn_compilar, chk_otimizar]), saida_status, abas_fases])
aba_estufa = widgets.VBox([widgets.HBox([widgets.VBox([s_relogio, s_umid, s_temp, s_luz]),
                                         widgets.VBox([s_res, s_clima, s_pragas])]),
                           widgets.HBox([btn_dia, btn_reset]), painel, diario])
display(HTML("<h3>🌱 CasaScript Studio — Estufa</h3>"))
display(criar_abas([aba_compilador, aba_estufa], ["🛠️ Compilador", "🌱 Estufa"]))
ao_compilar(None)

## Bateria de testes (RF6)

In [17]:
# 🧪 CÉLULA 16 — Bateria de testes automáticos

def rodar_testes(casos):
    """Cada caso = (nome, código-fonte, resultado esperado: "ok", "léxica", "sintática" ou "semântica")."""
    linhas, aprovados = [], 0
    for nome, fonte, esperado in casos:
        res = compilar(fonte)
        obtido = "ok" if res.ok else res.fase_erro
        passou = (obtido == esperado)
        aprovados += passou
        linhas.append(("✅" if passou else "❌", nome, esperado, obtido,
                       "" if res.ok else res.mensagens[0].splitlines()[0][:100]))
    tabela(["", "Caso", "Esperado", "Obtido", "Primeira mensagem"], linhas,
           titulo=f"🧪 {aprovados}/{len(casos)} testes passaram")

CASOS_DE_TESTE = [
    # corretos
    ("Regra simples",              'QUANDO pragas ENTAO ligar(ventilador) FIM',                                      "ok"),
    ("Horário válido",             'QUANDO relogio >= 18:30 ENTAO ajustar(lampada_cultivo, 70) FIM',                "ok"),
    ("Regra com SENAO",            'QUANDO umidade_solo < 30 ENTAO irrigar(irrigador, 10) SENAO desligar(irrigador) FIM', "ok"),
    ("Ações novas + horário",      'QUANDO relogio > 12:00 E temperatura > 30 ENTAO ventilar(ventilador, 90); irrigar(gotejador, 5) FIM', "ok"),
    # léxicos
    ("Caractere inválido",         'QUANDO temperatura @ 30 ENTAO ligar(ventilador) FIM',                           "léxica"),
    ("Horário 25:00",              'QUANDO relogio >= 25:00 ENTAO ligar(lampada_cultivo) FIM',                      "léxica"),
    ("Horário 18:75",              'QUANDO relogio >= 18:75 ENTAO ligar(lampada_cultivo) FIM',                      "léxica"),
    # sintáticos
    ("Faltou o FIM",               'QUANDO pragas ENTAO ligar(ventilador)',                                          "sintática"),
    ("SENAO vazio",                'QUANDO pragas ENTAO ligar(ventilador) SENAO FIM',                                "sintática"),
    ("ENTAO sem ação",             'QUANDO pragas ENTAO FIM',                                                        "sintática"),
    # semânticos
    ("Sensor inexistente",         'QUANDO chuva ENTAO fechar(janela_teto) FIM',                                     "semântica"),
    ("Horário × número",           'QUANDO relogio >= 18 ENTAO ligar(lampada_cultivo) FIM',                          "semântica"),
    ("Erro no SENAO (dispositivo)",'QUANDO pragas ENTAO ligar(ventilador) SENAO abrir(ventilador) FIM',              "semântica"),
]
rodar_testes(CASOS_DE_TESTE)

,Caso,Esperado,Obtido,Primeira mensagem
✅,Regra simples,ok,ok,
✅,Horário válido,ok,ok,
✅,Regra com SENAO,ok,ok,
✅,Ações novas + horário,ok,ok,
✅,Caractere inválido,léxica,léxica,"🔤 ERRO LÉXICO na linha 1, coluna 20: o caractere '@' não pertence à CasaScript."
✅,Horário 25:00,léxica,léxica,"🔤 ERRO LÉXICO na linha 1, coluna 19: '25:00' não é um horário válido (use 00:00 a 23:59)."
✅,Horário 18:75,léxica,léxica,"🔤 ERRO LÉXICO na linha 1, coluna 19: '18:75' não é um horário válido (use 00:00 a 23:59)."
✅,Faltou o FIM,sintática,sintática,"🌳 ERRO SINTÁTICO na linha 1, coluna 37: encontrei o fim do programa, mas esperava: ;, FIM, SENAO"
✅,SENAO vazio,sintática,sintática,"🌳 ERRO SINTÁTICO na linha 1, coluna 45: encontrei 'FIM', mas esperava: um nome"
✅,ENTAO sem ação,sintática,sintática,"🌳 ERRO SINTÁTICO na linha 1, coluna 21: encontrei 'FIM', mas esperava: um nome"


## Conclusão
O compilador funciona com as mudanças pedidas: horário, SENAO, tabela da estufa com os avisos novos, as duas otimizações novas e o SENAO no runtime. Todos os testes passaram.

A parte mais chata foi o horário. No começo o lexer quebrava o `18:30` em três tokens, até a gente lembrar da prioridade dos tokens. A borda de descida foi mais tranquila, porque é só o contrário da borda de subida.